# Caderno de aprendizagem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/00_aprendizagem.ipynb)

Este caderno vem **antes** de cada notebook do TCC. Os notebooks do TCC fazem a pesquisa em si, para as 370
estações. Aqui você aprende, com exemplos pequenos, cada ideia e cada ferramenta que eles usam, para chegar lá
sabendo o que está acontecendo.

### Como o caderno está organizado

Um capítulo para cada notebook do TCC, na mesma ordem. Cada capítulo tem dois tipos de item:

- **Resolvido**: o exemplo vem pronto, com a explicação antes e os comentários no código. Rode, leia e **mude os
  valores** para ver o que acontece;
- **Para pensar e resolver**: perguntas para responder com código ou com texto, na célula vazia logo abaixo.

As respostas dos itens "para pensar e resolver" são a pauta das reuniões com o orientador.

*Capítulos disponíveis: 1 e 2. Os capítulos 3 a 7 serão acrescentados junto com os notebooks 03 a 07.*

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula só define DE ONDE os dados serão lidos.
# No Colab, os dados são lidos direto do GitHub (não precisa baixar nada).
# No computador do orientador, dentro da pasta do projeto, ela usa a pasta dados/ local.
from pathlib import Path

if Path("../dados").exists():
    DADOS = "../dados/"                     # rodando dentro da pasta do projeto
else:
    DADOS = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/dados/"

print("Os dados serão lidos de:", DADOS)

In [ ]:
import pandas as pd

diaria = pd.read_csv(DADOS + "chuva_diaria_destaque.csv", dtype={"cod": str})
diaria["data"] = pd.to_datetime(diaria["data"])
diaria["ano"] = diaria["data"].dt.year
diaria["mes"] = diaria["data"].dt.month
sh = diaria[diaria["cod"] == "2454012"]          # Santa Helena
print("Dias de chuva diária de Santa Helena:", len(sh))

---

# Capítulo 1. Python com chuva

**Prepara para:** [01_python_basico](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/01_python_basico.ipynb)

### 1.1 Resolvido: a maior sequência de dias secos

Um **veranico** é uma sequência de dias sem chuva no meio da estação chuvosa, e é uma das maiores causas de perda
na soja. O código abaixo percorre os dias de janeiro de 2012 em Santa Helena e conta a maior sequência de dias
secos (chuva menor que 1 mm). A ideia: um contador que cresce a cada dia seco e volta a zero quando chove.

In [ ]:
jan2012 = sh[(sh["ano"] == 2012) & (sh["mes"] == 1)]

sequencia = 0            # dias secos seguidos até agora
maior = 0                # a maior sequência encontrada
for chuva in jan2012["chuva_mm"]:
    if chuva < 1.0:
        sequencia = sequencia + 1      # mais um dia seco
    else:
        sequencia = 0                  # choveu: a sequência recomeça
    if sequencia > maior:
        maior = sequencia              # guarda o recorde

print("Maior sequência de dias secos em jan/2012:", maior, "dias")
print("Chuva total do mês:", round(jan2012["chuva_mm"].sum(), 1), "mm")

### 1.2 Resolvido: transformar o código em função

O mesmo cálculo dentro de uma função, para usar em qualquer mês e em qualquer estação.

In [ ]:
def maior_veranico(chuvas, limiar=1.0):
    """Maior número de dias seguidos com chuva abaixo do limiar (mm)."""
    sequencia = 0
    maior = 0
    for chuva in chuvas:
        if chuva < limiar:
            sequencia = sequencia + 1
        else:
            sequencia = 0
        if sequencia > maior:
            maior = sequencia
    return maior


for ano in [2010, 2011, 2012]:
    dados = sh[(sh["ano"] == ano) & (sh["mes"] == 1)]
    print("Janeiro de", ano, ":", maior_veranico(dados["chuva_mm"]), "dias")

### 1.3 Para pensar e resolver

a) Use `maior_veranico` para achar o maior veranico de **dezembro a fevereiro** da safra 2011/2012 em Santa Helena
(dezembro de 2011 + janeiro e fevereiro de 2012). Dica: filtre as datas com
`sh[(sh["data"] >= "2011-12-01") & (sh["data"] <= "2012-02-29")]`.

b) Mude o limiar para 5 mm. O resultado muda muito? Por que alguém usaria 5 mm em vez de 1 mm?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 2. pandas: somar, agrupar e comparar

**Prepara para:** [02_dados_chuva](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/02_dados_chuva.ipynb)

### 2.1 Resolvido: chuva anual de uma estação, com `groupby`

`groupby("ano")` separa a tabela em pedaços, um por ano; `["chuva_mm"].sum()` soma a chuva de cada pedaço. O
resultado é uma série com um valor por ano.

In [ ]:
anual = sh[sh["ano"] <= 2019].groupby("ano")["chuva_mm"].sum()

print("Média:", round(anual.mean()), "mm")
print("Desvio padrão:", round(anual.std()), "mm")
print("Coeficiente de variação:", round(anual.std() / anual.mean() * 100, 1), "%")
anual.round(0).tail()

### 2.2 Resolvido: anomalia

A **anomalia** é quanto um ano ficou acima ou abaixo da média. Em porcentagem: (ano − média) ÷ média × 100. É a
ideia que está por trás do SPI, que você vai ver no capítulo 5.

In [ ]:
anomalia = (anual - anual.mean()) / anual.mean() * 100

print("Os 3 anos mais secos (anomalia em %):")
print(anomalia.sort_values().head(3).round(0))
print()
print("Os 3 anos mais chuvosos (anomalia em %):")
print(anomalia.sort_values(ascending=False).head(3).round(0))

### 2.3 Resolvido: juntar duas tabelas com `merge`

Muitas vezes a informação está em duas tabelas: a chuva numa, o nome da estação noutra. `merge` junta as duas pela
coluna que elas têm em comum (aqui, `cod`).

In [ ]:
est = pd.read_csv(DADOS + "estacoes_pr.csv", dtype={"cod": str})
media = diaria[diaria["ano"] <= 2019].groupby(["cod", "ano"])["chuva_mm"].sum().groupby("cod").mean().reset_index()

juntas = media.merge(est[["cod", "municipio", "alt_m"]], on="cod")
juntas.round(0)

### 2.4 Para pensar e resolver

a) Calcule a anomalia anual de **Cascavel** e diga se os 3 anos mais secos de Cascavel são os mesmos de Santa
Helena. Por que podem não ser?

b) O coeficiente de variação da chuva **anual** de Santa Helena é bem menor que o CV dos meses de inverno que você
vai ver no notebook 02. Por que a chuva de um ano inteiro varia menos que a de um mês?

In [ ]:
# escreva sua resposta aqui

*Escreva aqui a resposta do item b.*